Analysis

In [ ]:
import pandas as pd
import numpy as np
# from scipy import stats

df = pd.read_csv("delhi_metro_cleaned.csv")
print("Data loaded!Shape:",df.shape)
df.head(3)

Data loaded!Shape: (285, 8)


,station_id,station_name,dist_km,metro_line,opened_year,layout,latitude,longitude
0,1.0,Shaheed Sthal(First Station),0.0,Red Line,2019.0,Elevated,28.670611,77.415582
1,2.0,Hindon River,1.0,Red Line,2019.0,Elevated,28.878965,77.415483
2,3.0,Arthala,2.5,Red Line,2019.0,Elevated,28.676999,77.391892


Merge:


In [2]:
stations = df[['station_id', 'station_name', 'metro_line', 'dist_km']].copy()
layout_info = df[['station_name', 'layout', 'latitude', 'longitude']].copy()

merged_df = pd.merge(stations, layout_info, on='station_name', how='inner')
print("Inner Merge Shape:", merged_df.shape)
merged_df.head(3)

Inner Merge Shape: (333, 7)


,station_id,station_name,metro_line,dist_km,layout,latitude,longitude
0,1.0,Shaheed Sthal(First Station),Red Line,0.0,Elevated,28.670611,77.415582
1,2.0,Hindon River,Red Line,1.0,Elevated,28.878965,77.415483
2,3.0,Arthala,Red Line,2.5,Elevated,28.676999,77.391892


Left Join:

In [ ]:
left_merged = pd.merge(stations, layout_info, on='station_name', how='left')
print("Left Join Shape:", left_merged.shape)
left_merged.head(3)

Left Join Shape: (333, 7)


,station_id,station_name,metro_line,dist_km,layout,latitude,longitude
0,1.0,Shaheed Sthal(First Station),Red Line,0.0,Elevated,28.670611,77.415582
1,2.0,Hindon River,Red Line,1.0,Elevated,28.878965,77.415483
2,3.0,Arthala,Red Line,2.5,Elevated,28.676999,77.391892


 Groupby:

In [9]:
line_count = df.groupby('metro_line')['station_name'].count().reset_index()
line_count.columns = ['metro_line', 'station_count']
line_count = line_count.sort_values('station_count', ascending=False)
print(line_count)

           metro_line  station_count
1           Blue Line             49
8           Pink Line             38
12        Yellow Line             37
11        Voilet Line             34
10           Red Line             29
6        Magenta Line             25
0           Aqua Line             21
4          Green Line             21
9         Rapid Metro             11
2    Blue Line Branch              8
7         Orange Line              6
3           Gray Line              3
5   Green Line Branch              3


In [13]:
line_stats = df.groupby('metro_line')['dist_km'].agg(
    ['mean', 'max', 'min', 'sum']).reset_index()
line_stats.columns = ['metro_line', 'avg_dist', 'max_dist', 'min_dist', 'total_dist']
print(line_stats.round(2))

           metro_line  avg_dist  max_dist  min_dist  total_dist
0           Aqua Line     13.35      27.1       0.0       280.4
1           Blue Line     26.14      52.7       0.0      1281.1
2    Blue Line Branch      4.00       8.1       0.0        32.0
3           Gray Line      1.80       3.9       0.0         5.4
4          Green Line     11.38      24.8       0.0       239.0
5   Green Line Branch      1.07       2.1       0.0         3.2
6        Magenta Line     17.66      33.1       0.0       441.4
7         Orange Line     10.57      20.8       0.0        63.4
8           Pink Line     28.77      52.6       0.0      1093.4
9         Rapid Metro      5.71      10.0       0.0        62.8
10           Red Line     16.56      32.7       0.0       480.2
11        Voilet Line     20.62      43.5       0.0       701.0
12        Yellow Line     21.46      45.7       0.0       794.1


In [12]:
year_count = df.groupby('opened_year')['station_name'].count().reset_index()
year_count.columns = ['opened_year', 'stations_opened']
year_count = year_count.dropna()
print(year_count)

    opened_year  stations_opened
0        2002.0                6
1        2003.0                3
2        2004.0               11
3        2005.0               28
4        2006.0                9
5        2008.0                3
6        2009.0               17
7        2010.0               54
8        2011.0               13
9        2013.0                5
10       2014.0                3
11       2015.0               13
12       2017.0               18
13       2018.0               64
14       2019.0               37


Map:

In [ ]:
line_color_map = {
    'Red Line': 'red', 'Blue Line': 'blue',
    'Blue Line Branch': 'blue', 'Yellow Line': 'yellow',
    'Green Line': 'green', 'Green Line Branch': 'green',
    'Voilet Line': 'violet', 'Pink Line': 'pink',
    'Magenta Line': 'magenta', 'Aqua Line': 'cyan',
    'Orange Line': 'orange', 'Rapid Metro': 'gray',
    'Gray Line': 'gray'
}
df['line_color'] = df['metro_line'].map(line_color_map)
df[['station_name', 'metro_line', 'line_color']].head()

,station_name,metro_line,line_color
0,Shaheed Sthal(First Station),Red Line,red
1,Hindon River,Red Line,red
2,Arthala,Red Line,red
3,Mohan Nagar,Red Line,red
4,Shyam Park,Red Line,red


In [15]:
layout_map = {'Elevated': 1, 'Underground': 2, 'At-Grade': 3, 'Unknown': 0}
df['layout_code'] = df['layout'].map(layout_map)
df[['layout', 'layout_code']].value_counts()

layout       layout_code
Elevated     1              213
Underground  2               68
At-Grade     3                3
Unknown      0                1
Name: count, dtype: int64

Pivot Table:

In [16]:
pivot1 = pd.pivot_table(df,
                        values='station_name',
                        index='metro_line',
                        columns='layout',
                        aggfunc='count',
                        fill_value=0)
pivot1

layout,At-Grade,Elevated,Underground,Unknown
metro_line,,,,
Aqua Line,0,21,0,0
Blue Line,1,44,4,0
Blue Line Branch,1,7,0,0
Gray Line,0,2,1,0
Green Line,0,21,0,0
Green Line Branch,1,2,0,0
Magenta Line,0,10,15,0
Orange Line,0,1,5,0
Pink Line,0,26,12,0


In [17]:
pivot2 = pd.pivot_table(df,
                        values='dist_km',
                        index='metro_line',
                        columns='opened_year',
                        aggfunc='mean',
                        fill_value=0).round(2)
pivot2.head()

opened_year,2002.0,2003.0,2004.0,2005.0,2006.0,2008.0,2009.0,2010.0,2011.0,2013.0,2014.0,2015.0,2017.0,2018.0,2019.0
metro_line,,,,,,,,,,,,,,,
Aqua Line,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,13.35
Blue Line,0.0,0.0,0.0,20.13,14.36,0.0,41.38,0.85,0.0,0.0,0.0,0.0,0.0,0.00,50.42
Blue Line Branch,0.0,0.0,0.0,0.00,0.00,0.0,0.00,3.48,7.3,0.0,0.0,0.0,0.0,0.00,0.00
Gray Line,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,1.80
Green Line,0.0,0.0,0.0,0.00,0.00,0.0,0.00,7.04,0.0,0.0,0.0,0.0,0.0,20.07,0.00


Crosstab:

In [18]:
cross1 = pd.crosstab(df['metro_line'], df['layout'])
cross1

layout,At-Grade,Elevated,Underground,Unknown
metro_line,,,,
Aqua Line,0,21,0,0
Blue Line,1,44,4,0
Blue Line Branch,1,7,0,0
Gray Line,0,2,1,0
Green Line,0,21,0,0
Green Line Branch,1,2,0,0
Magenta Line,0,10,15,0
Orange Line,0,1,5,0
Pink Line,0,26,12,0


In [19]:
cross_norm = pd.crosstab(df['metro_line'], df['layout'],
                         normalize='index').round(2)
cross_norm

layout,At-Grade,Elevated,Underground,Unknown
metro_line,,,,
Aqua Line,0.00,1.00,0.00,0.00
Blue Line,0.02,0.90,0.08,0.00
Blue Line Branch,0.12,0.88,0.00,0.00
Gray Line,0.00,0.67,0.33,0.00
Green Line,0.00,1.00,0.00,0.00
Green Line Branch,0.33,0.67,0.00,0.00
Magenta Line,0.00,0.40,0.60,0.00
Orange Line,0.00,0.17,0.83,0.00
Pink Line,0.00,0.68,0.32,0.00


Statistics

In [20]:
print("Std dev  :", round(df['dist_km'].std(), 2))
print("Variance :", round(df['dist_km'].var(), 2))
print("Skewness :", round(df['dist_km'].skew(), 4))
print("Kurtosis :", round(df['dist_km'].kurt(), 4))

Std dev  : 14.0
Variance : 196.08
Skewness : 0.531
Kurtosis : -0.6631


In [21]:
print("Percentiles:")
df['dist_km'].quantile([0.25, 0.50, 0.75, 0.90])

Percentiles:


0.25     7.3
0.50    17.4
0.75    28.8
0.90    40.7
Name: dist_km, dtype: float64

Probability:

In [27]:
total_layout=(df["layout"]).count()
print(total_layout)
total_elev=(df["layout"]=="Elevated").sum()
print(total_elev)
p_elevated = (df['layout'] == 'Elevated').sum() / len(df)
p_blue = (df['metro_line'] == 'Blue Line').sum() / len(df)
p_both = ((df['layout'] == 'Elevated') & (df['metro_line'] == 'Blue Line')).sum() / len(df)
p_cond = p_both / p_blue

print(f"P(Elevated)             : {p_elevated:.2%}")
print(f"P(Blue Line)            : {p_blue:.2%}")
print(f"P(Elevated & Blue Line) : {p_both:.2%}")
print(f"P(Elevated | Blue Line) : {p_cond:.2%}")

285
213
P(Elevated)             : 74.74%
P(Blue Line)            : 17.19%
P(Elevated & Blue Line) : 15.44%
P(Elevated | Blue Line) : 89.80%


In [28]:
pivot1.to_csv("pivot_line_layout.csv")
cross1.to_csv("crosstab_line_layout.csv")
print("Analysis complete & tables saved")

Analysis complete & tables saved
